In [3]:
%%time

# Import libraries
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import os

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    precision_recall_curve,
    roc_curve,
    auc,
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)



# Load dataset
df_model = pd.read_csv('transformed_daily_data/daily_data_5y_30k_features1.csv')


# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort before creating shifted target
df_model = df_model.sort_values(['ticker', 'timestamp']).copy()

# Target: Close
df_model['close']
df



# Remove final row per ticker where next_close does not exist
df_model = df_model[df_model['next_close'].notna()].copy()

# Train: 2021-2024
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test: 2025
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()

# Columns to exclude from features
exclude_cols = [
    'timestamp',
    'ticker',
    'target',
    'next_close'
]

feature_cols = [col for col in df_model.columns if col not in exclude_cols]

X_train = train_df[feature_cols]
y_train = train_df['target']

X_test = test_df[feature_cols]
y_test = test_df['target']




model = RandomForestRegressor(
    n_estimators=200,
    max_features=1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)


model.fit(X_train, y_train)


y_test_pred = model.predict(X_test)
#y_val_pred = model.predict(X_val)





NameError: name 'X_val' is not defined

In [5]:
y_test

872       0
873       1
874       0
875       0
876       0
         ..
478297    1
478298    0
478299    1
478300    0
478301    1
Name: target, Length: 99250, dtype: int64

In [4]:
y_test_pred

array([0.525, 0.6  , 0.485, ..., 0.54 , 0.505, 0.67 ])

# Fix

In [6]:
%%time

# =========================
# IMPORT LIBRARIES
# =========================

import os
import joblib

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import (
    train_test_split,
    TimeSeriesSplit,
    GridSearchCV,
    cross_val_score,
    KFold
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)


# =========================
# LOAD DATA
# =========================

df_model = pd.read_csv(
    'transformed_daily_data/daily_data_5y_30k_features1.csv'
)


# =========================
# PREPARE DATA
# =========================

# Ensure datetime formatting
df_model['timestamp'] = pd.to_datetime(df_model['timestamp'])

# Sort before creating shifted target
df_model = df_model.sort_values(
    ['ticker', 'timestamp']
).copy()


# =========================
# CREATE REGRESSION TARGET
# =========================

# Predict next day's closing price
df_model['next_close'] = (
    df_model.groupby('ticker')['close'].shift(-1)
)

# Remove final row per ticker
# because next_close does not exist
df_model = df_model[
    df_model['next_close'].notna()
].copy()


# =========================
# TRAIN / TEST SPLIT
# =========================

# Train: 2021-2024
train_df = df_model[
    (df_model['timestamp'] >= '2021-01-01') &
    (df_model['timestamp'] < '2025-01-01')
].copy()

# Test: 2025
test_df = df_model[
    (df_model['timestamp'] >= '2025-01-01') &
    (df_model['timestamp'] < '2026-01-01')
].copy()


# =========================
# FEATURE SELECTION
# =========================

exclude_cols = [
    'timestamp',
    'ticker',
    'next_close'
]

feature_cols = [
    col for col in df_model.columns
    if col not in exclude_cols
]


X_train = train_df[feature_cols]
y_train = train_df['next_close']

X_test = test_df[feature_cols]
y_test = test_df['next_close']


# =========================
# MODEL
# =========================

model = RandomForestRegressor(
    n_estimators=200,
    max_features=1,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)


# =========================
# TRAIN MODEL
# =========================

model.fit(X_train, y_train)


# =========================
# PREDICTIONS
# =========================

y_test_pred = model.predict(X_test)


# =========================
# EVALUATION METRICS
# =========================

mae = mean_absolute_error(y_test, y_test_pred)
mse = mean_squared_error(y_test, y_test_pred)
rmse = np.sqrt(mse)
medae = median_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print("\n===== REGRESSION METRICS =====")

print(f"MAE:    {mae:.4f}")
print(f"MSE:    {mse:.4f}")
print(f"RMSE:   {rmse:.4f}")
print(f"MedAE:  {medae:.4f}")
print(f"R²:     {r2:.4f}")


===== REGRESSION METRICS =====
MAE:    2.4252
MSE:    98.6126
RMSE:   9.9304
MedAE:  0.7816
R²:     0.9952
CPU times: user 2min 23s, sys: 5.34 s, total: 2min 28s
Wall time: 24.1 s


# Results

In [14]:
results_df = pd.DataFrame()
results_df['next_close'] = y_test
results_df['pred_next_close'] = y_test_pred
results_df

,next_close,pred_next_close
872,35.71,38.317787
873,36.49,36.009283
874,36.24,36.264673
875,36.00,36.132440
876,35.91,35.909405
...,...,...
478297,126.23,125.586397
478298,125.98,125.856655
478299,126.41,125.911720
478300,125.82,126.490858


In [18]:
test_df['pred_next_close'] = (y_test_pred).round(2)
test_df['pred_error'] = test_df['pred_next_close'] - test_df['next_close']
test_df['abs_pred_error'] = test_df['pred_error'].abs()
test_df

,timestamp,ticker,open,high,low,close,volume,vwap,transactions,close_vs_open_pct,...,MACD_signal,MACD_hist,rolling_volatility,close_lag1,close_lag2,return_lag1,next_close,pred_next_close,pred_error,abs_pred_error
872,2025-01-02,AA,38.165,39.0400,37.9000,37.99,2703886.0,38.3283,33119.0,-0.004585,...,-1.349609,-0.075438,0.014294,37.78,37.15,0.016958,35.71,38.32,2.61,2.61
873,2025-01-03,AA,37.950,37.9500,35.3752,35.71,7491799.0,35.9337,63814.0,-0.059025,...,-1.387444,-0.151340,0.029454,37.99,37.78,0.005558,36.49,36.01,-0.48,0.48
874,2025-01-06,AA,36.000,37.0750,35.9200,36.49,5983917.0,36.4794,58230.0,0.013611,...,-1.419583,-0.128553,0.033236,35.71,37.99,-0.060016,36.24,36.26,0.02,0.02
875,2025-01-07,AA,36.890,37.3000,35.7550,36.24,3381466.0,36.2093,39434.0,-0.017620,...,-1.447219,-0.110544,0.032951,36.49,35.71,0.021843,36.00,36.13,0.13,0.13
876,2025-01-08,AA,35.780,36.0300,34.7500,36.00,3775303.0,35.5672,41012.0,0.006149,...,-1.471115,-0.095584,0.030724,36.24,36.49,-0.006851,35.91,35.91,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478297,2025-12-24,ZTS,123.100,125.6899,123.0600,125.49,2369020.0,125.1484,29115.0,0.019415,...,-1.891613,1.082639,0.008772,123.54,123.78,-0.001939,126.23,125.59,-0.64,0.64
478298,2025-12-26,ZTS,125.160,126.3200,124.8000,126.23,3226934.0,125.8979,45295.0,0.008549,...,-1.604182,1.149727,0.008752,125.49,123.54,0.015784,125.98,125.86,-0.12,0.12
478299,2025-12-29,ZTS,126.160,126.8500,125.5607,125.98,4465924.0,126.0785,51439.0,-0.001427,...,-1.321637,1.130177,0.008154,126.23,125.49,0.005897,126.41,125.91,-0.50,0.50
478300,2025-12-30,ZTS,125.550,127.6000,125.4500,126.41,3230541.0,126.5667,45221.0,0.006850,...,-1.047095,1.098168,0.007307,125.98,126.23,-0.001981,125.82,126.49,0.67,0.67
